# A Machine worker in Colab (W20)

This notebook makes the Colab VM a **worker** of the mesh whose coordinator runs on your laptop.
Nothing about the job semantics changes; what changes is the transport's three assumptions
(see `networking/README.md`, "Beyond one filesystem"):

* the worker **only dials out** — Colab cannot be dialled, and does not need to be;
* the coordinator is reached through a **tunnel** you start on the laptop (ngrok / bore /
  `ssh -R`); TLS is end to end, so the tunnel sees ciphertext and is trusted with nothing;
* the worker verifies the coordinator by **certificate identity** under the CA pin rather
  than by hostname, and every job's inputs and results travel **over the wire** in
  digest-checked chunks instead of being paths on a shared disk.

You need three things from the laptop before running: the tunnel's public `HOST:PORT`, the
**bundle** that `python main.py mesh --role issue --cert-name colab-1 --cert-dir … --bundle-dir …`
wrote (exactly `ca.crt`, `colab-1.crt`, `colab-1.key` — never the CA key), and a coordinator
started with `--artifacts wire`. The runbook is `colab/README.md`.

In [ ]:
#@title Parameters
# COORDINATOR: the tunnel's public endpoint, as the laptop printed it
COORDINATOR = "0.tcp.eu.ngrok.io:12345"   #@param {type:"string"}
# CERT_NAME: the name the bundle was issued for; EXPECT: the coordinator's certificate name
CERT_NAME = "colab-1"                     #@param {type:"string"}
EXPECT = "coordinator"                    #@param {type:"string"}
MODES = "dfs,bfs,astar,beam,rewritedfs"   #@param {type:"string"}
# SWARM: serve W16 work units (snapshots) instead of agenda search jobs
SWARM = False                             #@param {type:"boolean"}
# RUN_SECONDS: stop after this long (Colab sessions are finite)
RUN_SECONDS = 10800                       #@param {type:"integer"}
REPO = "https://github.com/StrangeTcy/cat_theo_machine"
BRANCH = "master"                         #@param {type:"string"}
PKI_DIR = "/content/hyge-pki"             # outside the repository tree on purpose: the guard refuses keys inside it
ARTIFACT_DIR = "/content/hyge-artifacts"

In [ ]:
#@title 1. The repository and its two dependencies
import os, subprocess
if not os.path.isdir("/content/cat_theo_machine"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/cat_theo_machine"], check=True)
else:
    subprocess.run(["git", "-C", "/content/cat_theo_machine", "pull", "--ff-only"], check=True)
subprocess.run(["pip", "-q", "install", "pyyaml", "gmpy2"], check=True)
print(subprocess.run(["git", "-C", "/content/cat_theo_machine", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
#@title 2. The bundle (upload the three files the laptop issued)
import os
os.makedirs(PKI_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)
wanted = ["ca.crt", CERT_NAME + ".crt", CERT_NAME + ".key"]
missing = [name for name in wanted if not os.path.exists(os.path.join(PKI_DIR, name))]
if missing:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join(PKI_DIR, os.path.basename(name)), "wb") as handle:
            handle.write(data)
    os.chmod(os.path.join(PKI_DIR, CERT_NAME + ".key"), 0o600)
present = sorted(os.listdir(PKI_DIR))
print("bundle:", present)
assert "ca.key" not in present, "the CA key must never leave the laptop; delete it here and re-issue the bundle"
assert all(os.path.exists(os.path.join(PKI_DIR, name)) for name in wanted), "still missing: %s" % missing

In [ ]:
#@title 3. Run the worker (this cell streams the worker's log until it stops)
import os, subprocess, sys
cmd = [sys.executable, "-u", "-m", "cat_theo_machine.main", "mesh",
       "--role", "worker", "--coordinator", COORDINATOR,
       "--cert-dir", PKI_DIR, "--cert-name", CERT_NAME,
       "--no-inbound", "--verify", "identity", "--expect", EXPECT,
       "--artifacts", "wire", "--artifact-dir", ARTIFACT_DIR,
       "--modes", MODES, "--run-seconds", str(RUN_SECONDS)]
if SWARM:
    cmd.append("--swarm")
print(" ".join(cmd))
env = dict(os.environ, PYTHONPATH="/content")
proc = subprocess.Popen(cmd, cwd="/content", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if line.startswith("[mesh:") or "Traceback" in line or "Error" in line:
        print(line.rstrip())
proc.wait()
print("worker exited", proc.returncode)

## What you should see, and what it means when you do not

```
[mesh:colab-1] no listener (--no-inbound): outbound connections only
[mesh:colab-1] artifacts travel over the wire; local root /content/hyge-artifacts
[mesh:colab-1] joined 0.tcp.eu.ngrok.io:12345 as colab-1 (identity check: coordinator)
[mesh:colab-1] booting the machine for the first job (one boot per peer)
[mesh:colab-1] booted in 21.2s
```

* `peer identity mismatch on join: expected coordinator, got …` — the laptop's coordinator
  runs under another `--cert-name`; set `EXPECT` to that name.
* `CERTIFICATE_VERIFY_FAILED` either way — the bundle was issued by a different CA than the
  one the coordinator uses (a fresh `--cert-dir` on the laptop makes a fresh CA); re-issue.
* `Connection refused` / timeout — the tunnel is down or points at the wrong local port.
* `no certificate named 'x' in /content/hyge-pki` — `CERT_NAME` does not match the bundle.

The worker's own ledger is written to `/content/hyge-pki/peer-<name>.json` when it stops;
the coordinator's `ledger.json` on the laptop has the authoritative account of every job.